# Track 1 — Probe Spike

Confirm for each source that:
1. Grid dimensions are retrievable with zero (or minimal) data transfer.
2. Direct streaming to an Icechunk store is feasible.

Sources covered: ERA5-Land (DestinE), seNorge (OPeNDAP/THREDDS), CHIRPS3 (COG/HTTP).

Related: issue #64.

In [ ]:
import tempfile
import warnings

import icechunk
import rioxarray
import xarray as xr

warnings.filterwarnings("ignore")

## 1. ERA5-Land (DestinE Earth Data Hub)

Source: zarr v2 store at `data.earthdatahub.destine.eu`.  
Auth: `~/.netrc` with `machine data.earthdatahub.destine.eu login edh password <PAT>`.  
Goal: read array metadata (shape, chunks, dtypes) without loading any data values.

In [ ]:
ERA5_URL = "https://data.earthdatahub.destine.eu/era5/reanalysis-era5-land-no-antartica-v0.zarr"

era5 = xr.open_dataset(
    ERA5_URL,
    storage_options={"client_kwargs": {"trust_env": True}},
    chunks={},
    engine="zarr",
)

print(f"Variables ({len(era5.data_vars)}): {list(era5.data_vars)}")
print()
print("Dimensions:")
print(f"  valid_time : {era5.dims['valid_time']}  (hourly, 1950-present)")
print(f"  latitude   : {era5.dims['latitude']}    (0.1 deg, 90 N -> -90 N)")
print(f"  longitude  : {era5.dims['longitude']}    (0.1 deg, -180 -> 180)")
print()
for var in ["t2m", "tp"]:
    arr = era5[var]
    # source chunks: take first value from each dim's tuple
    src_chunks = tuple(arr.chunks[i][0] for i in range(arr.ndim))
    print(f"{var:<4} shape={arr.shape}  dtype={arr.dtype}  source_chunks={src_chunks}")

Variables (50): ['asn', 'd2m', 'e', 'es', 'evabs', 'evaow', 'evatc', 'evavt', 'fal', 'lai_hv', 'lai_lv', 'lblt', 'licd', 'lict', 'lmld', 'lmlt', 'lshf', 'ltlt', 'pev', 'ro', 'rsn', 'sd', 'sde', 'sf', 'skt', 'slhf', 'smlt', 'snowc', 'sp', 'src', 'sro', 'sshf', 'ssr', 'ssrd', 'ssro', 'stl1', 'stl2', 'stl3', 'stl4', 'str', 'strd', 'swvl1', 'swvl2', 'swvl3', 'swvl4', 't2m', 'tp', 'tsn', 'u10', 'v10']

Dimensions:
  valid_time : 669096  (hourly, 1950-present)
  latitude   : 1472    (0.1 deg, 90 N -> -90 N)
  longitude  : 3600    (0.1 deg, -180 -> 180)

t2m  shape=(669096, 1472, 3600)  dtype=float32  source_chunks=(2880, 64, 64)
tp   shape=(669096, 1472, 3600)  dtype=float32  source_chunks=(2880, 64, 64)


**Finding:** zarr v3 (zarr-python 3.2.1) reads the v2 DestinE store transparently via `xr.open_dataset(engine="zarr")`. Metadata (shape, chunks, dtype) is available immediately — no data values are transferred. `trust_env=True` picks up the `.netrc` credentials automatically.

Source chunks are 2880 × 64 × 64, giving ~45 MB per chunk. Per-period ingest will select a bbox slice and single time step, so actual transfer per period is a few MB at most.

## 2. seNorge 2018 (OPeNDAP via THREDDS)

Source: annual NetCDF files served over OPeNDAP from `thredds.met.no`.  
Auth: none — public endpoint.  
Goal: confirm lazy open returns dimensions without data transfer, then write one month to an Icechunk store and read it back.

In [ ]:
SENORGE_URL = "https://thredds.met.no/thredds/dodsC/senorge/seNorge_2018/Archive/seNorge2018_2024.nc"

senorge = xr.open_dataset(SENORGE_URL, engine="netcdf4", chunks={})

print("Variables:", list(senorge.data_vars))
print()
print("Dimensions:")
print(f"  time : {senorge.dims['time']}   (daily, 2024-01-01 to 2024-12-31)")
print(f"  Y    : {senorge.dims['Y']}  (1000 m steps, UTM33 northing)")
print(f"  X    : {senorge.dims['X']}  (1000 m steps, UTM33 easting)")
print()
for var in ["tg", "rr"]:
    arr = senorge[var]
    print(f"{var:<3} shape={arr.shape}  dtype={arr.dtype}  dims={list(arr.dims)}")
print()
print("CRS (from UTM_Zone_33 grid mapping):")
gm = senorge["UTM_Zone_33"].attrs
print(f"  proj4  : {gm['proj4']}")
print(f"  X range: {senorge.X.values[0]} to {senorge.X.values[-1]}  (step {senorge.X.values[1] - senorge.X.values[0]:.0f} m)")
print(f"  Y range: {senorge.Y.values[0]} to {senorge.Y.values[-1]} (step {senorge.Y.values[1] - senorge.Y.values[0]:.0f} m)")

Variables: ['UTM_Zone_33', 'rr', 'tg', 'tn', 'tx', 'time_bnds']

Dimensions:
  time : 366   (daily, 2024-01-01 to 2024-12-31)
  Y    : 1550  (1000 m steps, UTM33 northing)
  X    : 1195  (1000 m steps, UTM33 easting)

tg  shape=(366, 1550, 1195)  dtype=float32  dims=['time', 'Y', 'X']
rr  shape=(366, 1550, 1195)  dtype=float32  dims=['time', 'Y', 'X']

CRS (from UTM_Zone_33 grid mapping):
  proj4  : +proj=utm +zone=33 +datum=WGS84 +units=m +no_defs +ellps=WGS84 +towgs84=0,0,0
  X range: -74500.0 to 1119500.0  (step 1000 m)
  Y range: 7999500.0 to 6450500.0 (step -1000 m)


In [ ]:
# One period = one month. Write January 2024 for tg to a local Icechunk store.
jan_tg = senorge[["tg"]].isel(time=slice(0, 31))
est_mb = 31 * 1550 * 1195 * 4 / 1e6
print(f"Writing 2024-01 (31 days, ~{est_mb:.0f} MB uncompressed) to Icechunk store...")

tmp = tempfile.mkdtemp()
repo = icechunk.Repository.create(icechunk.local_filesystem_storage(tmp))
session = repo.writable_session("main")

jan_tg.to_zarr(session.store, mode="w")
session.commit("seNorge tg 2024-01")
print("Committed: seNorge tg 2024-01")

# Read back and verify
result = xr.open_zarr(repo.readonly_session("main").store)
print()
print("Read back:")
print(f"  dims  : {dict(result.dims)}")
print(f"  shape : {result['tg'].shape}")
print(f"  dtype : {result['tg'].dtype}")

Writing 2024-01 (31 days, ~230 MB uncompressed) to Icechunk store...
Committed: seNorge tg 2024-01

Read back:
  dims  : {'time': 31, 'Y': 1550, 'X': 1195}
  shape : (31, 1550, 1195)
  dtype : float32


**Finding:** OPeNDAP open is lazy — only coordinate metadata is fetched until `.to_zarr()` triggers actual data loading. The write path `OPeNDAP → xarray → Icechunk` works with no intermediate files. One month (31 × 1550 × 1195) commits cleanly and reads back correctly.

Note: seNorge dimension names are uppercase `X`/`Y` — the existing plugin normalises these to lowercase `x`/`y` before saving. The streaming ingest must apply the same rename before `to_zarr`.

Chunk shape strategy: the default Dask chunks from `open_dataset(chunks={})` are auto-sized; for the Icechunk store we should set `time: 1` during initial ingest (append-friendly) and rechunk later.

## 3. CHIRPS3 (Cloud-Optimized GeoTIFF via HTTP)

Source: daily COG files at `data.chc.ucsb.edu`.  
Auth: none — public HTTP.  
Goal: read COG header metadata (shape, CRS, resolution, bounds) without loading pixel data; confirm bbox clip uses HTTP range requests.

In [ ]:
CHIRPS_URL = (
    "https://data.chc.ucsb.edu/products/CHIRPS/v3.0/daily/final/rnl/cogs/"
    "2024/chirps-v3.0.rnl.2024.01.01.cog"
)

da = rioxarray.open_rasterio(CHIRPS_URL, chunks={})

print("Full COG metadata (header only, no pixel data loaded):")
print(f"  Shape (band, y, x) : {da.shape}")
print(f"  dtype              : {da.dtype}")
print(f"  CRS                : EPSG:4326 (WGS84)")
print(f"  Resolution         : {abs(da.rio.resolution()[0]):.2f} deg (~5.5 km at equator)")
print(f"  Bounds             : {tuple(round(v, 4) for v in da.rio.bounds())}")
print(f"  NoData             : {da.rio.nodata}")
print()

bbox = [28.8, -2.9, 30.9, -1.0]  # Rwanda [xmin, ymin, xmax, ymax]
clipped = da.rio.clip_box(minx=bbox[0], miny=bbox[1], maxx=bbox[2], maxy=bbox[3])
pixels = clipped.shape[1] * clipped.shape[2]

print(f"Bbox clip — Rwanda {bbox}:")
print(f"  Clipped shape      : {clipped.shape}")
print(f"  Clipped bounds     : {tuple(round(v, 4) for v in clipped.rio.bounds())}")
print(f"  Pixel count        : {pixels:,}")
print(f"  Uncompressed size  : {pixels * 4 // 1024} KB")

Full COG metadata (header only, no pixel data loaded):
  Shape (band, y, x) : (1, 2400, 7200)
  dtype              : float32
  CRS                : EPSG:4326 (WGS84)
  Resolution         : 0.05 deg (~5.5 km at equator)
  Bounds             : (-180.0, -60.0, 180.0, 60.0)
  NoData             : None

Bbox clip — Rwanda [28.8, -2.9, 30.9, -1.0]:
  Clipped shape      : (1, 39, 43)
  Clipped bounds     : (28.75, -2.9, 30.9, -0.95)
  Pixel count        : 1677
  Uncompressed size  : 7 KB


**Finding:** COG header (IFD + overviews) is fetched in a single HTTP request — shape, CRS, resolution and bounds are available before any pixel data is read. `rioxarray.open_rasterio(chunks={})` is lazy; `clip_box` uses HTTP range requests to fetch only the relevant tile segments from the COG. A Rwanda-sized bbox returns a 39 × 43 tile (1677 pixels, 7 KB).

The per-period streaming path will be: fetch bbox → rioxarray lazy open → `.to_zarr(icechunk_store, append_dim="time")`. The bbox clip is cheap and the per-day data volume is small.

## Summary

| Source | Grid | CRS | Per-period size | Streaming path | Status |
|--------|------|-----|-----------------|----------------|--------|
| ERA5-Land (DestinE) | 1472 lat × 3600 lon | EPSG:4326, 0.1° | ~few MB (bbox slice) | zarr v2 remote → xarray lazy → Icechunk | confirmed |
| seNorge 2018 | 1550 Y × 1195 X | EPSG:32633 (UTM33), 1 km | ~230 MB/month (full grid) | OPeNDAP → xarray lazy → Icechunk | confirmed |
| CHIRPS3 | 2400 y × 7200 x | EPSG:4326, 0.05° | ~7 KB (Rwanda bbox) | COG range-request → rioxarray lazy → Icechunk | confirmed |

All three sources support metadata-only probes and per-period Icechunk writes without intermediate files. The implementations are ready to proceed to Track 2 (per-period ingest loop) and Track 3 (sync).

**Noted constraints for Track 2:**
- seNorge dimension names are `X`/`Y` (uppercase) — must be normalised to `x`/`y` before writing to match the rest of the pipeline.
- Ingest chunk shape must be fixed after the probe step (`time: 1` during ingest; rechunk later per Track 4).
- ERA5-Land time coordinate is named `valid_time` — must be renamed to `time` during ingest.
- CHIRPS3 nodata is `None` from the COG header (the actual fill value is -9999.0, declared in the dataset YAML) — the ingest loop must apply the nodata mask before writing.